# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
print("Current dir:", os.getcwd())
print(os.listdir("."))

Current dir: /content
['.config', 'sample_data']


In [4]:
!git clone https://github.com/Reva1404/flyrank-ml-internship-starter.git
os.chdir("flyrank-ml-internship-starter")
print(os.getcwd())

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 237 (delta 104), reused 70 (delta 70), pack-reused 97 (from 2)
Receiving objects: 100% (237/237), 1.89 MiB | 5.11 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/flyrank-ml-internship-starter


In [6]:
!python scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
import pandas as pd
import os

df = pd.read_csv("outputs/refresh_queue.csv")
print(df.shape)
df.head()

(30000, 28)


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.734212,random_forest,0.783472,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_d6570c51c9bd,client_3fdba35f04,81.603243,random_forest,0.849842,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
2,3,content_6aa43079fb0c,client_3fdba35f04,81.544618,random_forest,0.789490,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
3,4,content_72e800a9c214,client_3fdba35f04,81.169731,random_forest,0.776297,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.957565,random_forest,0.816010,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [8]:
# The queue already has final_reason_codes and suggested_action from the pipeline.
# We rename them to match our playbook's language.
df["reason_codes"] = df["final_reason_codes"]
df["suggested_action_model"] = df["suggested_action"]
df["confidence"] = df["confidence"]  # already high/medium/low

ranked_queue = df.sort_values("final_refresh_score", ascending=False)
print(ranked_queue["reason_codes"].value_counts().head(10))
print(ranked_queue["suggested_action_model"].value_counts())
ranked_queue.head(20)[["content_id","final_refresh_score","confidence",
                        "suggested_action_model","reason_codes"]]

reason_codes
general_refresh_review                                                                                           7708
page_one_decay_risk                                                                                              2371
declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate     1959
declining_with_demand|model_decline_risk                                                                         1556
declining_with_demand                                                                                            1343
declining_with_demand|model_decline_risk|visible_model_opportunity                                               1148
general_refresh_review|model_decline_risk                                                                         808
declining_with_demand|page_one_decay_risk|low_ctr_visible_page|visible_model_opportunity|ctr_review_candidate     800
low_engagement_visible_page|engagement_revi

,content_id,final_refresh_score,confidence,suggested_action_model,reason_codes
0,content_1f080331fa2b,81.734212,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...
1,content_d6570c51c9bd,81.603243,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
2,content_6aa43079fb0c,81.544618,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
3,content_72e800a9c214,81.169731,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
4,content_e04eb9549989,80.957565,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
5,content_b69288c5e701,80.798090,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
6,content_9b6df29f7889,80.650656,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
7,content_ba6f9dfcbca1,80.432641,medium,refresh,declining_with_demand|model_decline_risk|visib...
8,content_4d76cdb3387b,80.428403,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
9,content_b4f35d640b1c,80.428243,medium,refresh,declining_with_demand|model_decline_risk|visib...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** This ranked queue supports a content editor deciding which pages to review
first each week, given limited review capacity. It is decision-support only.

**Where it stops being valid:**
- Pages under `content_age_days < 90` (not enough history)
- Any page flagged for consolidation, seasonality, or SERP-layout change (this model cannot tell those apart from real decline)
- Clients with very sparse traffic (impressions_90d near the minimum threshold) — low signal, high noise

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [9]:
high_conf_review = ranked_queue[ranked_queue["confidence"] == "high"].head(20)
high_conf_review[["content_id","final_refresh_score","suggested_action_model","reason_codes",
                   "impressions_90d","trend_direction"]]

,content_id,final_refresh_score,suggested_action_model,reason_codes,impressions_90d,trend_direction
0,content_1f080331fa2b,81.734212,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,down
2,content_6aa43079fb0c,81.544618,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,down
3,content_72e800a9c214,81.169731,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,down
5,content_b69288c5e701,80.798090,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,down
6,content_9b6df29f7889,80.650656,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,down
10,content_bb6ebb5ec8c8,80.359854,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2621,down
11,content_1b51115391a3,80.255676,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3221,down
13,content_20ddbdfbbd90,79.933106,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,5541,down
14,content_b1d593faf9c6,79.918772,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,2655,down
15,content_91fefd1726b3,79.685897,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2748,down


**Before acting on any row, a human must check:**
1. Does the page's actual content match the reason code given? (Read it — don't trust the label blindly.)
2. Is this a case of consolidation (a sibling page absorbed the traffic) rather than real decline?
3. Is the sample size (impressions/sessions) large enough to trust?

**Never automate:**
- Publishing edits directly from this queue
- Deleting or de-indexing any page
- Any action on `low` confidence rows without manual verification

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [10]:
base_rate = df["is_declining_label"].mean()
top50_precision = df.sort_values("final_refresh_score", ascending=False).head(50)["is_declining_label"].mean()
print(f"Base rate: {base_rate:.3f}")
print(f"Precision@50 (current): {top50_precision:.3f}")

Base rate: 0.542
Precision@50 (current): 0.980


**Retrain triggers:**
- Precision@50 on a fresh month drops below ~0.5 (roughly halfway between base rate and current score)
- Reason-code distribution shifts drastically (e.g. `general_review` jumps from <5% to >30%)
- A full quarter passes without retraining, regardless of metrics

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
os.makedirs("work/outputs", exist_ok=True)

export_cols = ["content_id","client_id","final_refresh_score","confidence",
               "suggested_action_model","reason_codes","impressions_90d",
               "trend_direction","is_declining_label"]

ranked_queue[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("Exported:", os.path.exists("work/outputs/action_playbook_queue.csv"))

Exported: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.